In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
from scipy.stats import binomtest

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
    

In [ ]:
def filter_output(v):
    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]
    
    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["Gene"].isin(orth["Gene name"])]
    v = v[~v["Gene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["Gene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    
    #Remove olfactory receptors and HLA genes
    v = v[~v["Gene"].isin(olfs + hlas + ugt)]

    return v

In [ ]:
v = pd.read_csv("AccelEvol_Missense_Sim1000000.txt", sep = "\t")
v.columns = ["Gene"] + list(v.columns)[1:]
v = filter_output(v[((v["Num Hum Var"] >= 3) | (v["Num Chp Var"] >= 3)) & ((v["Hum Sum PhyloP"] > 0) | (v["Chp Sum PhyloP"] > 0))])
v = v.sort_values("Sum p-value")
v["Sum p-value"] = np.maximum(v["Sum p-value"]*2, 1/1000000)
v["FDR"] = fdrcorrection(v["Sum p-value"])[1]
np.max(v["PhyloP Difference"])

In [ ]:
k = []
for index, row in v.iterrows():
    if row["PhyloP Difference"] < 0 and row["FDR"] < 0.05:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["PhyloP Difference"] < 0 and row["Sum p-value"] < 0.05:
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["PhyloP Difference"] > 0 and row["FDR"] < 0.05:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["PhyloP Difference"] > 0 and row["Sum p-value"] < 0.05:
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v["Acceleration"] = k

In [ ]:
v = v.drop(["PhyloP L2FC", "Total_Sims", "Better_Sims_L2FC", "Better_Sims_Sum", "L2FC p-value"], axis = 1)
v.columns = ["Protein gene symbol", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR", "Acceleration"]
v.to_csv("Supplemental_Tables/Supplemental_Table1.csv", sep = ",", index = False)

In [ ]:
v[(v["FDR"] < 0.05) & (v["PhyloP Difference"] > 0)].shape

In [ ]:
ha = v[(v["Sum p-value"] < 0.05) & (v["PhyloP Difference"] > 0)].shape[0]
ca = v[(v["Sum p-value"] < 0.05) & (v["PhyloP Difference"] < 0)].shape[0]
binomtest(ha, ha + ca)

In [ ]:
plt.figure(figsize=(4, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [389, 693 - 389], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of nominally accelerated proteins", size = 12)
plt.ylim(0, 450)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("More human acceleration in proteins", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 405, 5             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "**",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

sns.barplot(y=["Human", "Chimp"], x=[389, 693 - 389], hue=["Human", "Chimp"], palette=palette, alpha=0.8)

plt.xlabel("Number of nominally accelerated proteins")
plt.xlim(0, 450)
#plt.xticks(size=12)
#plt.yticks(size=12)
plt.title("More human acceleration in proteins")
plt.legend([], [], frameon=False)

# --- significance bracket (horizontal) ---
y1, y2 = 0, 1        # bar positions (Human, Chimp)
x, h = 405, 5        # x-position of bracket + horizontal offset

plt.plot([x, x + h, x + h, x], [y1, y1, y2, y2], lw=1.2, c='black')

plt.text(x + 2*h, (y1 + y2) * 0.5, "**", ha='left', va='center', fontsize=14)

plt.show()

In [ ]:
sig = []
#Also filtered out sites that were variable in apes or variable across human populations
for index, row in v.iterrows():
    if row["FDR"] < 0.05 and row["PhyloP Difference"] < 0:
        sig.append("Chimp accelerated")
    elif row["FDR"] < 0.05 and row["PhyloP Difference"] > 0:
        sig.append("Human accelerated")
    else:
        sig.append("Not significant")
v["Category"] = sig

In [ ]:
v[v["Category"] == "Chimp accelerated"].shape

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)
v["-Log$_{10}$(FDR)"] = -np.log10(v["FDR"])
v = v.sort_values("Category")
sns.scatterplot(data = v, x = "PhyloP Difference", y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})
#plt.ylim(-2, 30)
plt.xlim(floor(np.min(v["PhyloP Difference"]) + 0.1*np.min(v["PhyloP Difference"])), floor(np.max(v["PhyloP Difference"]) + 0.1*np.max(v["PhyloP Difference"])))
#plt.vlines(0.25, -2, 30, linestyles="dashed", colors="k")
#plt.vlines(-0.25, -2, 30, linestyles="dashed", colors="k")
plt.hlines(-np.log10(0.05), floor(np.min(v["PhyloP Difference"]) + 0.1*np.min(v["PhyloP Difference"])), floor(np.max(v["PhyloP Difference"]) + 0.1*np.max(v["PhyloP Difference"])), linestyles="dashed", colors="k")
plt.legend([], [], frameon = False)
plt.xlabel("Difference in PhyloP")
plt.title("Accelerated protein evolution")

In [ ]:
#Muscle Cell Differentiation is really interesting given smooth/non-smooth differences, especially ACTN3
#Estrogen signaling is also cool
#Idk what to make of rRNA Metabolic Process...
#We should restrict only to those that continue to be significant after removing the top 3
v = pd.read_csv("AccelEvol_Missense_HPO_Sim_All1000000.txt", sep = "\t")
v = v[((v["Num Hum Var"] >= 10) | (v["Num Chp Var"] >= 10)) & ((v["Hum Sum PhyloP"] > 0) | (v["Chp Sum PhyloP"] > 0))]
v["Sum p-value"] = v["Better_Sims_Sum"]/v["Total_Sims"]
v["Sum p-value"] = np.maximum(v["Sum p-value"]*2, 1/1000000)
v = v.sort_values("Sum p-value")

v["FDR"] = fdrcorrection(v["Sum p-value"])[1]

v2 = pd.read_csv("AccelEvol_Missense_HPO_Sim_rm3_All1000000.txt", sep = "\t")
v2 = v2[((v2["Num Hum Var"] >= 10) | (v2["Num Chp Var"] >= 10)) & ((v2["Hum Sum PhyloP"] > 0) | (v2["Chp Sum PhyloP"] > 0))]
v2["Sum p-value"] = v2["Better_Sims_Sum"]/v2["Total_Sims"]
v2["Sum p-value"] = np.maximum(v2["Sum p-value"]*2, 1/1000000)
v2 = v2.sort_values("Sum p-value")

v2["FDR"] = fdrcorrection(v2["Sum p-value"])[1]
v2[v2["Geneset"].isin(v[v["FDR"] < 0.05]["Geneset"])]

In [ ]:
v2 = v2[["Geneset", "Sum p-value"]]
v2.columns = ["Term", "p-value removing three most accel. genes"]


In [ ]:
v = v[["Geneset", "Hum Sum PhyloP", "Num Hum Var", "Chp Sum PhyloP", "Num Chp Var", "PhyloP Difference", "Sum p-value", "FDR"]]
v.columns = ["Term", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]
v["Ontology"] = np.repeat("HPO", v.shape[0])
v = v[["Ontology", "Term", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "FDR"]]


In [ ]:
v_gobp = v.set_index("Term").join(v2.set_index("Term")).copy()

In [ ]:
v_hpo = v.set_index("Term").join(v2.set_index("Term")).copy()

In [ ]:
v = pd.concat([v_gobp, v_hpo])
k = []
for index, row in v.iterrows():
    if row["Difference in PhyloP-weighted sum"] < 0 and row["FDR"] < 0.05:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] < 0 and row["p-value"] < 0.05:
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["FDR"] < 0.05:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["p-value"] < 0.05:
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v["Acceleration"] = k
v = v.sort_values("p-value")
v.to_csv("Supplemental_Tables/Supplemental_Table2.csv")

In [ ]:
vx = vx.sort_values("p-value")
vx.to_csv("Supplemental_Tables/Supplemental_Table2.csv")

In [ ]:
z = pd.read_csv("HumChp_Mis_Final_Tri447.bed", sep = "\t", header = None)
z2 = pd.read_csv("HumChp_Mis_Final_ChpDer_Tri447.bed", sep = "\t", header = None)
z["Derived"] = np.repeat("Human", z.shape[0])
z2["Derived"] = np.repeat("Chimp", z2.shape[0])
z = pd.concat([z, z2])
z["Position"] = z[0] + ":" + z[2].astype(str)
z = z[["Position", 3, 5, 11, "Derived"]]
z = z[z[5] != "."]
z[5] = z[5].astype(float)

In [ ]:
back_p = z[z["Derived"] == "Human"].shape[0]/(z[z["Derived"] == "Human"].shape[0] + z[z["Derived"] == "Chimp"].shape[0])

In [ ]:
back_p

In [ ]:
from scipy.stats import mannwhitneyu as mwu

sig = v[(v["FDR"] < 0.05) & (v["Geneset"].isin(v2[v2["Sum p-value"] < 0.05]["Geneset"]))]
bi = []
mwul = []
mean_hum = []
mean_chp = []
std_hum = []
std_chp = []
new_df = []
for index, row in sig.iterrows():
    bi.append(binomtest(row["Num Hum Var"], row["Num Chp Var"] + row["Num Hum Var"], p = back_p, alternative = "greater").pvalue)
    z2 = z[z[3].isin(d_HPO[row["Geneset"]])].copy()
    z2["PhyloP Non"] = np.maximum(0, z2[5])
    h = z2[z2["Derived"] == "Human"]["PhyloP Non"]

    for i in h:
        new_df.append([row["Geneset"], i, "Human"])
    c = z2[z2["Derived"] == "Chimp"]["PhyloP Non"]
    for i in c:
        new_df.append([row["Geneset"], i, "Chimp"])
    mh = np.mean(z2[z2["Derived"] == "Human"]["PhyloP Non"].astype(float))
    sh = np.std(z2[z2["Derived"] == "Human"]["PhyloP Non"].astype(float))
    mc = np.mean(z2[z2["Derived"] == "Chimp"]["PhyloP Non"].astype(float))
    sc = np.std(z2[z2["Derived"] == "Chimp"]["PhyloP Non"].astype(float))
    mean_hum.append(mh)
    mean_chp.append(mc)
    std_hum.append(sh)
    std_chp.append(sc)
    mwul.append(mwu(z2[z2["Derived"] == "Human"]["PhyloP Non"].astype(float), z2[z2["Derived"] == "Chimp"]["PhyloP Non"].astype(float), alternative = "greater")[1])
sig["Binomial p-value"] = bi
sig["MWU p-value"] = mwul
sig["Mean hum"] = mean_hum
sig["Mean chp"] = mean_chp
sig["Std hum"] = std_hum
sig["Std chp"] = std_chp

sig[sig["MWU p-value"] < 0.05].shape

In [ ]:
df_hpo = pd.DataFrame(new_df)
df_hpo.columns = ["Term", "PhyloP", "Species"]

In [ ]:
df_bp = pd.DataFrame(new_df)
df_bp.columns = ["Term", "PhyloP", "Species"]

In [ ]:
df_bp["Term"] = [" ".join([y.lower() if y != "Muscle" and y != "Golgi" else y for y in x.split(" ")[:-1]]) for x in df_bp['Term']]

In [ ]:
fig, ax = plt.subplots(figsize = (9, 4), dpi = 450)
df = pd.concat([df_hpo, df_bp])
sns.boxplot(data = df, x = "Term", y = "PhyloP", hue = "Species", palette = palette)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Non-negative PhyloP", size = 24)
plt.xlabel("Ontology term", size = 24)
plt.title("Shift toward higher PhyloP for human-derived subs.", size = 24)
plt.xticks(size = 14)
plt.yticks(size = 14)
plt.legend([], [], frameon = False)

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

sig = []
#Also filtered out sites that were variable in apes or variable across human populations
for index, row in v.iterrows():
    if row["FDR"] < 0.05 and row["PhyloP Difference"] < 0:
        sig.append("Chimp accelerated")
    elif row["FDR"] < 0.05 and row["PhyloP Difference"] > 0:
        sig.append("Human accelerated")
    else:
        sig.append("Not significant")
v["Category"] = sig

v["-Log$_{10}$(FDR)"] = -np.log10(v["FDR"])
v = v.sort_values("Category")
sns.scatterplot(data = v, x = "PhyloP Difference", y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})
#plt.ylim(-2, 30)
plt.xlim(-125, 500)
#plt.vlines(0.25, -2, 30, linestyles="dashed", colors="k")
#plt.vlines(-0.25, -2, 30, linestyles="dashed", colors="k")
plt.hlines(-np.log10(0.05), -125, 500, linestyles="dashed", colors="k")
plt.legend([], [], frameon = False)
plt.xlabel("Difference in PhyloP")
plt.title("Human Phenotype Ontology terms")

In [ ]:
z = pd.read_csv("HumChp_Mis_Final_Tri447.bed", sep = "\t", header = None)
z2 = pd.read_csv("HumChp_Mis_Final_ChpDer_Tri447.bed", sep = "\t", header = None)
z["Derived"] = np.repeat("Human", z.shape[0])
z2["Derived"] = np.repeat("Chimp", z2.shape[0])
z = pd.concat([z, z2])
z["Position"] = z[0] + ":" + z[2].astype(str)
z = z[["Position", 3, 5, 11, "Derived"]]
z = z[z[5] != "."]
z[5] = z[5].astype(float)
z.index = z["Position"]

In [ ]:
z2 = z[z[3].isin(d_BP["Muscle Cell Differentiation (GO:0042692)"])]
from scipy.stats import mannwhitneyu as mwu
z2["PhyloP Non"] = np.maximum(0, z2[5])
mwu(z2[z2["Derived"] == "Human"]["PhyloP Non"].astype(float), z2[z2["Derived"] == "Chimp"]["PhyloP Non"].astype(float), alternative = "greater")

In [ ]:
x = pd.read_csv("../Myriad_RNA/SKM_Filtered.txt", sep = "\t")
x2 = x[x["gene"].isin(d_BP["Muscle Cell Differentiation (GO:0042692)"])]


In [ ]:
plt.figure(figsize=(6, 4), dpi=600)
sns.kdeplot(data = z2, hue = "Derived", x = "PhyloP Non", common_norm = False, fill = True, palette = palette)
plt.xlabel("Non-negative PhyloP")
plt.title("Muscle cell differentiation")